# ਸਬਕ 09 - ਮੈਟਾਕੋਗਨੀਸ਼ਨ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ


## ਸੈਟਅੱਪ

ਇਹ ਨੋਟਬੁੱਕ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ Metacognition ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਨੂੰ ਦਰਸਾਉਂਦੀ ਹੈ।

**ਪੂਰਵ-ਆਵਸ਼ਕਤਾਵਾਂ:**
- Azure OpenAI ਦੀ ਤੈਨਾਤੀ ਵਾਤਾਵਰਣ ਵੈਰੀਏਬਲਾਂ ਰਾਹੀਂ ਸੰਰਚਿਤ ਹੋਵੇ
- Azure CLI ਲਈ ਲੌਗਇਨ ਕੀਤਾ ਹੋਇਆ ਹੋਵੇ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ਮੈਟਾਕੋਗਨੀਸ਼ਨ ਕੀ ਹੈ?

ਮੈਟਾਕੋਗਨੀਸ਼ਨ **ਸੋਚ ਬਾਰੇ ਸੋਚਣਾ** ਹੈ। ਏਆਈ ਏਜੰਟਾਂ ਦੇ ਸੰਦਰਭ ਵਿੱਚ, ਇਸਦਾ ਅਰਥ ਉਹਨਾਂ ਏਜੰਟਾਂ ਬਣਾਉਣਾ ਹੈ ਜੋ ਇਹ ਕਰ ਸਕਣ:

- **ਆਤਮ-ਚਿੰਤਨ** ਆਪਣੇ ਨਤੀਜੇ ਅਤੇ ਤਰਕ ਕਰਨ ਦੀ ਪ੍ਰਕਿਰਿਆ 'ਤੇ
- **ਗਲਤੀਆਂ ਪਛਾਣਨਾ** ਅਤੇ ਚੁਪਚਾਪ ਨਾਕਾਮ ਹੋਣ ਦੀ ਥਾਂ ਸੁਚੱਜੇ ਢੰਗ ਨਾਲ ਬਹਾਲ ਹੋਣਾ
- **ਮੂਲਾਂਕਣ** ਕਿ ਉਹਨਾਂ ਦੇ ਜਵਾਬ ਪੂਰੇ ਅਤੇ ਮਦਦਗਾਰ ਹਨ ਜਾਂ ਨਹੀਂ
- **ਅਨੁਕੂਲ ਹੋਣਾ** ਆਪਣੀ ਰਣਨੀਤੀ ਜਦੋਂ ਇੱਕ ਪ੍ਰਾਰੰਭਿਕ ਤਰੀਕਾ ਕੰਮ ਨਾ ਕਰੇ (ਜਿਵੇਂ ਕਿ ਬੈਕਅਪ ਸਿਸਟਮ 'ਤੇ ਵਾਪਸ ਜਾਣਾ)

ਇੱਕ ਮੈਟਾਕੋਗਨੀਸ਼ਨ ਏਜੰਟ ਸਿਰਫ ਸਵਾਲਾਂ ਦੇ ਜਵਾਬ ਨਹੀਂ ਦਿੰਦਾ — ਇਹ ਆਪਣੇ ਆਪ ਦੀ ਕਾਰਗੁਜ਼ਾਰੀ ਦੀ ਨਿਗਰਾਨੀ ਕਰਦਾ ਹੈ ਅਤੇ ਤੁਰੰਤ ਸੋਧ ਕਰ ਲੈਂਦਾ ਹੈ।


## ਪ੍ਰਾਇਮਰੀ ਅਤੇ ਬੈਕਅੱਪ ਟੂਲ

ਮੇਟਾਕੌਗਨੀਸ਼ਨ ਦੀ ਇੱਕ ਆਮ ਪੈਟਰਨ ਹੈ **ਫਾਲਬੈਕ ਰਣਨੀਤੀ**. ਏਜੰਟ ਪਹਿਲਾਂ ਇੱਕ ਪ੍ਰਾਇਮਰੀ ਟੂਲ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ; ਜੇ ਇਹ ਅਸਫਲ ਰਹਿੰਦਾ ਹੈ (ਜਿਵੇਂ ਕਿ 404 ਐਰਰ), ਤਾਂ ਏਜੰਟ ਅਸਫਲਤਾ ਨੂੰ ਪਛਾਣਦਾ ਹੈ ਅਤੇ ਪਾਰਦਰਸ਼ਤਾ ਨਾਲ ਇੱਕ ਬੈਕਅੱਪ ਟੂਲ 'ਤੇ ਸਵਿੱਚ ਕਰਦਾ ਹੈ.

ਇਹ ਵਾਸਤਵਿਕ-ਦੁਨੀਆ ਸਿਸਟਮਾਂ ਦੀਆਂ ਪ੍ਰਣਾਲੀਆਂ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ ਜਿੱਥੇ ਪ੍ਰਾਇਮਰੀ ਸੇਵਾਵਾਂ ਉਪਲਬਧ ਨਾ ਹੋ ਸਕਦੀਆਂ ਹਨ ਅਤੇ ਏਜੰਟਾਂ ਨੂੰ ਵਿਕਲਪ ਚੁਣਨ ਤੋਂ ਪਹਿਲਾਂ ਸਮੱਸਿਆ ਦੀ ਖੁਦ-ਤਸ਼ਖੀਸ ਕਰਨੀ ਪੈਂਦੀ ਹੈ.

ਹੇਠਾਂ ਅਸੀਂ ਦੋ ਫਲਾਈਟ ਲੁੱਕਅੱਪ ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ:
- **ਪ੍ਰਾਇਮਰੀ** — ਪੈਰਿਸ, ਟੋਕਿਓ, ਅਤੇ ਬਾਰਸਿਲੋਨਾ ਨੂੰ ਢੱਕਦਾ ਹੈ
- **ਬੈਕਅੱਪ** — ਬਰਲਿਨ, ਸਿਡਨੀ, ਅਤੇ ਨਿਊਯਾਰਕ ਸਿਟੀ ਨੂੰ ਢੱਕਦਾ ਹੈ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ਗਲਤੀ ਰਿਕਵਰੀ ਨਾਲ ਸਵ-ਪ੍ਰਤੀਬਿੰਬੀ ਏਜੰਟ

ਹੇਠਾਂ ਦਿੱਤੇ ਏਜੰਟ ਨੂੰ ਹੁਕਮ ਦਿੱਤਾ ਗਿਆ ਹੈ ਕਿ ਉਹ ਪਹਿਲਾਂ ਪ੍ਰਾਇਮਰੀ ਫਲਾਈਟ ਸਿਸਟਮ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰੋ, ਨਾਕਾਮੀਆਂ ਨੂੰ ਪਛਾਣੋ, ਅਤੇ ਖੁੱਲ੍ਹ ਕੇ ਬੈਕਅਪ ਸਿਸਟਮ ਵੱਲ ਵਾਪਸ ਜਾਓ। ਹਰ ਜਵਾਬ ਦੇ ਬਾਅਦ ਇਹ ਸੰਖੇਪ ਵਜੋਂ ਆਪਣੀ-ਆਤਮ-ਮੁਲਾਂਕਣ ਕਰਦਾ ਹੈ ਕਿ ਕੀ ਇਸਨੇ ਯੂਜ਼ਰ ਦੇ ਸਵਾਲ ਦਾ ਪੂਰਾ ਜਵਾਬ ਦਿੱਤਾ।


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ਸਵ-ਮੁਲਾਂਕਣ ਨਮੂਨਾ

ਮੈਟਾਕੌਗਨੀਸ਼ਨ ਦਾ ਇੱਕ ਹੋਰ ਪੱਖ **ਸਵ-ਮੁਲਾਂਕਣ** ਹੈ: ਇੱਕ ਵੱਖਰਾ ਏਜੰਟ (ਜਾਂ ਦੂਜੇ ਪਾਸ ਵਿੱਚ ਉਹੀ ਏਜੰਟ) ਇੱਕ ਜਵਾਬ ਦੀ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਸਹਾਇਕਤਾ ਲਈ ਸਮੀਖਿਆ ਕਰਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ `ResponseEvaluator` ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜੋ ਟ੍ਰੈਵਲ-ਏਜੰਟ ਦੇ ਉੱਤਰਾਂ ਨੂੰ ਤਿੰਨ ਮਾਪਦੰਡਾਂ 'ਤੇ ਸਕੋਰ ਕਰਦਾ ਹੈ।


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਕਿਵੇਂ **ਮੈਟਾਕੌਗਨਿਟਿਵ ਏਜੰਟ** Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਬਣਾਉਣੇ ਜਾ ਸਕਦੇ ਹੋ, ਇਹ ਸਿੱਖਿਆ:

- **ਆਤਮ-ਚਿੰਤਨ**: ਏਜੰਟ ਜੋ ਆਪਣੀ ਸੋਚ-ਵਿਚਾਰ ਦੀ ਨਿਗਰਾਨੀ ਕਰਦੇ ਹਨ ਅਤੇ ਸਪੱਸ਼ਟ ਤੌਰ 'ਤੇ ਦੱਸਦੇ ਹਨ ਕਿ ਕੀ ਹੋਇਆ।
- **ਫਾਲਬੈਕ ਨਾਲ ਏਰਰ ਰਿਕਵਰੀ**: ਇੱਕ ਪ੍ਰਾਈਮਰੀ + ਬੈਕਅਪ ਟੂਲ ਪੈਟਰਨ ਜਿਸ ਵਿੱਚ ਏਜੰਟ ਨਾਕਾਮੀਆਂ (ਉਦਾਹਰਣ ਵਜੋਂ, 404 ਗਲਤੀਆਂ) ਦਾ ਪਤਾ ਲਗਾਂਦਾ ਹੈ ਅਤੇ ਆਟੋਮੈਟਿਕ ਤੌਰ 'ਤੇ ਕਿਸੇ ਵਿਕਲਪੀ ਸਰੋਤ ਨੂੰ ਅਜ਼ਮਾਉਂਦਾ ਹੈ।
- **ਸਵੈ-ਮੁਲਾਂਕਣ**: ਇੱਕ ਵੱਖਰਾ ਇਵੈਲੂਏਟਰ ਏਜੰਟ ਜੋ ਜਵਾਬਾਂ ਨੂੰ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਸਹਾਇਕਤਾ ਲਈ ਸਕੋਰ ਕਰਦਾ ਹੈ।

ਇਹ ਪੈਟਰਨ ਏਜੰਟਾਂ ਨੂੰ ਹੋਰ ਮਜ਼ਬੂਤ, ਪਾਰਦਰਸ਼ੀ ਅਤੇ ਭਰੋਸੇਯੋਗ ਬਣਾਉਂਦੇ ਹਨ — ਉਤਪਾਦਨ ਤੈਨਾਤੀਆਂ ਲਈ ਸੰਕਟਮੁਕਤ ਗੁਣ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਅਸਪਸ਼ਟੀਕਰਨ:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਿੱਥੇ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਯਤਨ ਕਰਦੇ ਹਾਂ, ਫਿਰ ਵੀ ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਤਰੁੱਟੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਉਸ ਦੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਅਧਿਕਾਰਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਸੰਵੇਦਨਸ਼ੀਲ ਜਾਂ ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਦੇ ਕਾਰਨ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
